## Set the notebook working directory

Run the following cell first. It selects the Gadi checkout; edit `repo` for another location. Image display cells below load files through Python using this directory.


In [ ]:
import os
from pathlib import Path

# Change this path if your checkout is elsewhere.
repo = Path("/scratch/vp91") / os.environ["USER"] / "intro-to-mpi"
if not (repo / "notebooks").is_dir() or not (repo / "src").is_dir():
    raise FileNotFoundError(f"Repository not found at {repo}. Update repo above.")
os.chdir(repo)
print(f"Working directory: {Path.cwd()}")


## Relevant files

### 1. Header files

- [common.h](../src/common.h)

### 2. Source files

- [04-jacobi-blocking.cu](../src/04-jacobi-blocking.cu)
- [05-jacobi-overlap.cu](../src/05-jacobi-overlap.cu)
- [07-device-collectives.cu](../src/07-device-collectives.cu)
- [08-strided-halo.cu](../src/08-strided-halo.cu)
- [09-managed-memory.cu](../src/09-managed-memory.cu)
- [11-jacobi-rma.cu](../src/11-jacobi-rma.cu)
- [12-stream-aware-mpi.cu](../src/12-stream-aware-mpi.cu)

### 3. PBS scripts

- [04-jacobi-blocking.pbs](../job-script/04-jacobi-blocking.pbs)
- [05-jacobi-overlap.pbs](../job-script/05-jacobi-overlap.pbs)
- [07-device-collectives.pbs](../job-script/07-device-collectives.pbs)
- [08-strided-halo.pbs](../job-script/08-strided-halo.pbs)
- [09-managed-memory.pbs](../job-script/09-managed-memory.pbs)
- [11-stream-aware-mpi.pbs](../job-script/11-stream-aware-mpi.pbs)

Build the executables before submission in an allocated compute environment, using the same modules as the PBS scripts. The scripts assume `/scratch/vp91/$USER/intro-to-mpi/build/bin`; update their executable paths if your checkout differs.


# GPU-aware MPI best practices

This checklist summarizes the practical rules used throughout the repository.
It is specific to MPI communication involving CUDA memory, not a general MPI
or CUDA guide.


Companion to [21-gpu-aware-mpi-best-practices](../docs/source/21-gpu-aware-mpi-best-practices.rst). Shell and PBS examples are documentation cells: this notebook does not execute GPU programs or submit jobs. Run GPU diagnostics only in an allocated compute job.

## Buffer ownership and ordering

* Complete the producer CUDA kernel before MPI reads a send buffer.
* Do not modify a send buffer until its nonblocking MPI request completes.
* Do not launch a consumer kernel until the receive or RMA operation completes.
* Use CUDA events and explicit stream dependencies when work is not on the
  default stream.
* Keep the communication buffer alive for the entire MPI request or RMA epoch.

## Support and transport

* Verify CUDA-aware support in the loaded MPI build, not only at compile time.
* Test point-to-point, collectives, RMA, derived datatypes, and managed memory
  separately; support is not necessarily uniform across operations.
* Treat a successful device-pointer call as API support, not proof of GPUDirect
  RDMA or zero-copy transport.
* Record the CUDA, MPI, UCX or OFI, driver, GPU, NIC, and topology versions for
  reproducible measurements.

## Rank, GPU, and NIC placement

* Map ranks using node-local placement rather than assuming global rank equals
  GPU number.
* Check GPU-to-GPU and GPU-to-NIC locality with topology tools on allocated
  compute nodes.
* Avoid accidental GPU oversubscription and measure the effect of NUMA and PCIe
  locality.

## Communication patterns

* Start with contiguous device buffers and explicit point-to-point operations.
* Use nonblocking operations only when buffer ownership and MPI progress are
  clear.
* Use packing kernels for strided device data when derived-datatype support or
  performance is uncertain.
* Compare explicit halo exchange with neighborhood collectives, persistent
  requests, and RMA only after establishing a correct baseline.
* Treat GPU-initiated communication as a separate programming model; standard
  MPI calls are normally launched by the CPU.

## Progress and overlap

* Do not assume MPI progresses while a GPU kernel occupies the device.
* Measure communication and kernel timelines independently before claiming
  overlap.
* Test whether the MPI library needs CPU polling or an asynchronous progress
  thread for the selected transport.
* Use warm-up iterations and synchronize at measurement boundaries.

## Memory choices

* Prefer ordinary device allocations for the clearest CUDA-aware MPI baseline.
* Test managed memory separately; page migration and registration can dominate
  communication time.
* Reuse buffers when possible to reduce allocation and memory-registration
  overhead.
* Keep host staging buffers pinned when explicit staging is required.

## Correctness experiments

For every new GPU-aware MPI pattern, test at least:

* one rank and multiple ranks;
* one node and multiple nodes;
* small and large messages;
* default and non-default CUDA streams;
* first-use and warmed-up iterations; and
* a result compared with a CPU or host-staged reference.

Report the MPI implementation and transport configuration with performance
results. A result that is correct on one cluster does not establish portability
to another CUDA-aware MPI stack.

## Apply the checklist to Jacobi

For the blocking solver, finish each GPU update before the next halo exchange. For the overlapped solver, keep outgoing boundaries unchanged until sends complete and wait for incoming halos before computing the boundary points. Interior work may run independently if it does not touch communication buffers.

For the stream-ordering example, the producer event is necessary before MPI reads the send buffer. After a supported blocking receive returns, the consumer kernel can be launched directly: an additional receive-ready CUDA event is unnecessary.

GPU-backed RMA windows require support beyond ordinary CUDA-aware point-to-point communication. General CUDA support does not establish support for `MPI_Get` on GPU windows. RMA also needs an iteration protocol that prevents a rank from reading values its neighbour has not yet produced.

## Practical review

Use these metadata commands with the same modules used to build and run the application:

```bash
module list
mpirun --version
ompi_info --parsable --all | grep mpi_built_with_cuda_support:value
ompi_info --param osc all
```

Expected interpretation: `value:true` reports build-level CUDA support. The OSC listing reports available one-sided components; neither output guarantees GPU-memory window support. Use the [diagnostics notebook](18-cuda-aware-mpi-diagnostics.ipynb) to collect runtime evidence in a compute job.

## Exercises

1. Annotate every send and receive buffer in a Jacobi iteration with the operation that releases it for reuse.
2. Explain why posting `MPI_Irecv` is insufficient to launch a consumer kernel. Identify the required completion call.
3. Compare a small distributed grid against a CPU or host-staged reference. Check all owned cells, excluding duplicate halos, rather than relying on one sample.
4. Design same-node and cross-node experiments with small and large messages. Record correctness separately from transport and timing evidence.
5. Explain why a zero centre sample after a small number of Jacobi iterations need not imply failure: boundary influence can propagate by at most one grid cell per iteration.

Expected outcomes: identify producer completion, request completion, and buffer lifetime separately; require evidence for overlap and GPUDirect claims; document unsupported cases rather than treating them as successful tests. A single-rank test must have defined behaviour even when its neighbours are `MPI_PROC_NULL`.